# GraphFNet — PCQM-Contact (LRGB Link Prediction)

PCQM-Contact is a link prediction task from the Long Range Graph Benchmark.
The model must predict which atom pairs are in 3D contact (within 4Å)
based only on the 2D molecular graph structure.

**Key differences from Peptides:**
- Task: link prediction (edge-level), not graph classification
- Metric: MRR, Hits@1, Hits@3, Hits@10 (not AP)
- Model outputs node embeddings → dot-product edge scoring
- Loss: BCEWithLogitsLoss on positive/negative edge pairs
- No graph-level pooling needed

In [ ]:
!pip uninstall -y torch-geometric
!pip install torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import time
import csv
import os
from tqdm import tqdm
from torch_geometric.datasets import LRGBDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_batch, to_dense_adj

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Utilities

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def log_results_to_csv(filepath, row_dict):
    file_exists = os.path.isfile(filepath)
    with open(filepath, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

## Dataset Loading

In [ ]:
def load_pcqm_contact(batch_size=32):
    """
    PCQM-Contact from LRGB.
    Each graph has:
      data.x              — node features (9 integer atom features)
      data.edge_index     — molecular graph edges
      data.edge_attr      — edge features (3 bond features)
      data.edge_label_index — [2, E_eval] pairs to score
      data.edge_label       — [E_eval] binary labels (1=contact, 0=non-contact)
    """
    train_dataset = LRGBDataset(root='./data/LRGB', name='PCQM-Contact', split='train')
    val_dataset   = LRGBDataset(root='./data/LRGB', name='PCQM-Contact', split='val')
    test_dataset  = LRGBDataset(root='./data/LRGB', name='PCQM-Contact', split='test')

    print(f'Train: {len(train_dataset)} graphs')
    print(f'Val:   {len(val_dataset)} graphs')
    print(f'Test:  {len(test_dataset)} graphs')

    # Inspect a sample
    s = train_dataset[0]
    print(f'\nSample graph:')
    print(f'  Nodes: {s.num_nodes}')
    print(f'  Edges (molecular): {s.edge_index.size(1)}')
    print(f'  Node features: {s.x.shape}')
    print(f'  Edge label pairs: {s.edge_label_index.shape}')
    print(f'  Edge labels: {s.edge_label.shape}, pos={s.edge_label.sum().item()}')

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size)

    return train_loader, val_loader, test_loader

## Model Components
Identical to Peptides notebook — no changes to the backbone.

In [ ]:
class SimpleAtomEncoder(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(64, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(10, hidden_dim),
            nn.Embedding(3,  hidden_dim),
            nn.Embedding(10, hidden_dim),
        ])
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = x.long().clamp(min=0)
        out = sum(emb(x[..., i]) for i, emb in enumerate(self.embeddings))
        return self.proj(F.gelu(out))


class DenseGCNLayer(nn.Module):
    def __init__(self, hidden_dim, edge_dim=3):
        super().__init__()
        self.node_lin = nn.Linear(hidden_dim, hidden_dim)
        self.edge_lin = nn.Linear(edge_dim, 1)
        self.norm     = nn.LayerNorm(hidden_dim)

    def forward(self, x, A_norm, edge_attr_dense=None):
        if edge_attr_dense is not None:
            E   = torch.sigmoid(self.edge_lin(edge_attr_dense)).squeeze(-1)
            msg = torch.bmm(A_norm * E, x)
        else:
            msg = torch.bmm(A_norm, x)
        return self.norm(F.gelu(self.node_lin(msg)))


class SpectralMixMH(nn.Module):
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.filter_gen = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj   = nn.Linear(hidden_dim, hidden_dim)
        self.norm       = nn.LayerNorm(hidden_dim)

    def forward(self, x, U, mask):
        x_hat      = torch.bmm(U.transpose(1, 2), x)
        fil        = torch.sigmoid(self.filter_gen(x_hat))
        x_filtered = fil * x_hat
        x_out      = torch.bmm(U, x_filtered)
        x_out      = x_out * mask.unsqueeze(-1)
        return self.norm(self.out_proj(F.gelu(x_out)))


print('Model components defined.')

## GraphFNet Contact Model

Key difference from Peptides model:
- No graph-level pooling (AttentionPooling removed)
- No graph classifier
- Returns **node embeddings** [B, N, H]
- Separate edge scoring module uses dot product of node pairs

The edge scoring happens outside the backbone during training/evaluation.

In [ ]:
class GraphFNet_Contact(nn.Module):
    """
    GraphFNet backbone for link prediction.
    Returns node-level embeddings — no pooling, no graph classifier.
    Edge scores are computed externally via dot product of node pair embeddings.
    """
    def __init__(
        self,
        hidden_dim = 128,
        num_layers = 4,
        num_heads  = 4,
        lap_k      = 8,
        dropout    = 0.1,
        edge_dim   = 3,
    ):
        super().__init__()
        self.lap_k   = lap_k
        self.dropout = nn.Dropout(dropout)

        self.input_proj = SimpleAtomEncoder(hidden_dim)
        self.pe_encoder = nn.Linear(lap_k, hidden_dim)

        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'local':  DenseGCNLayer(hidden_dim, edge_dim=edge_dim),
                'global': SpectralMixMH(hidden_dim, num_heads),
                'gate':   nn.Linear(hidden_dim, hidden_dim),
                'norm':   nn.LayerNorm(hidden_dim),
            })
            for _ in range(num_layers)
        ])

        # Node-level projection head (maps H → H for edge scoring)
        self.node_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def compute_laplacian_basis(self, adj, mask):
        B, N, _ = adj.shape
        A_list, U_list = [], []

        for b in range(B):
            n      = int(mask[b].sum().item())
            adj_b  = adj[b, :n, :n]
            deg    = adj_b.sum(dim=1)
            deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)
            D_inv_sqrt   = torch.diag(deg_inv_sqrt)
            A_norm_b     = D_inv_sqrt @ adj_b @ D_inv_sqrt
            L_b = torch.eye(n, device=adj.device) - A_norm_b

            try:
                _, U_b = torch.linalg.eigh(L_b)
                # Sign canonicalization
                max_abs_idx = torch.abs(U_b).argmax(dim=0)
                signs = torch.sign(
                    U_b[max_abs_idx, torch.arange(U_b.size(1), device=adj.device)]
                )
                signs[signs == 0] = 1.0
                U_b = U_b * signs.unsqueeze(0)
            except Exception:
                U_b = torch.eye(n, device=adj.device)

            A_pad = F.pad(A_norm_b, (0, N-n, 0, N-n))
            U_pad = F.pad(U_b,      (0, N-n, 0, N-n))
            A_list.append(A_pad)
            U_list.append(U_pad)

        return torch.stack(A_list), torch.stack(U_list)

    def forward(self, data):
        """
        Returns:
            node_emb: [B, N, H] — dense node embeddings (padded)
            mask:     [B, N]    — True for real nodes
        """
        x, mask = to_dense_batch(data.x.float(), data.batch)
        adj     = to_dense_adj(
            data.edge_index, data.batch,
            max_num_nodes=x.size(1)
        )

        if data.edge_attr is not None:
            edge_attr_dense = to_dense_adj(
                data.edge_index, data.batch,
                edge_attr=data.edge_attr[:, :3].float(),
                max_num_nodes=x.size(1)
            )
        else:
            edge_attr_dense = None

        I   = torch.eye(adj.size(1), device=x.device).unsqueeze(0)
        adj = adj + I

        A_norm, U = self.compute_laplacian_basis(adj, mask)

        x = self.input_proj(x)

        k      = min(self.lap_k, U.size(-1))
        lap_pe = U[:, :, :k] * mask.unsqueeze(-1)
        if k < self.lap_k:
            lap_pe = F.pad(lap_pe, (0, self.lap_k - k))
        x = x + self.pe_encoder(lap_pe)

        for layer in self.layers:
            x_res    = x
            x_local  = layer['local'](x, A_norm, edge_attr_dense)
            x_global = layer['global'](x, U, mask)
            gate     = torch.sigmoid(layer['gate'](x))
            x_mix    = gate * x_local + (1 - gate) * x_global
            x        = layer['norm'](x_res + self.dropout(x_mix))

        x = x * mask.unsqueeze(-1)    # zero out padding
        node_emb = self.node_head(x)  # [B, N, H]

        return node_emb, mask


def score_edges(node_emb, edge_label_index, batch_vector):
    """
    Score candidate edges via dot product of node embedding pairs.

    Args:
        node_emb:         [B, N, H] dense node embeddings
        edge_label_index: [2, E] global node indices for candidate pairs
        batch_vector:     [total_nodes] batch assignment for each node

    Returns:
        scores: [E] dot-product scores for each candidate edge
    """
    # Build a flat node embedding tensor [total_nodes, H]
    # by selecting the correct positions from the dense batch
    B, N, H = node_emb.shape

    # Count nodes per graph to find intra-batch offsets
    nodes_per_graph = torch.bincount(batch_vector, minlength=B)  # [B]
    offsets = torch.cat([torch.tensor([0], device=node_emb.device),
                         nodes_per_graph.cumsum(0)[:-1]])         # [B]

    # Flatten node embeddings: only real nodes (not padding)
    # node_emb[b, :nodes_per_graph[b], :] for each b
    flat_emb_list = []
    for b in range(B):
        n = nodes_per_graph[b].item()
        flat_emb_list.append(node_emb[b, :n, :])    # [n, H]
    flat_emb = torch.cat(flat_emb_list, dim=0)       # [total_nodes, H]

    src_idx = edge_label_index[0]   # [E]
    dst_idx = edge_label_index[1]   # [E]

    src_emb = flat_emb[src_idx]     # [E, H]
    dst_emb = flat_emb[dst_idx]     # [E, H]

    scores = (src_emb * dst_emb).sum(dim=-1)  # [E] dot product
    return scores


print('GraphFNet_Contact model defined.')

## Evaluation — MRR and Hits@K

PCQM-Contact evaluation protocol:
- For each positive (contact) edge, rank it against all negative edges in the same graph
- Compute Mean Reciprocal Rank (MRR) and Hits@1, Hits@3, Hits@10

In [ ]:
@torch.no_grad()
def evaluate_contact(model, loader):
    """
    Evaluates GraphFNet_Contact on PCQM-Contact.
    Returns dict with MRR, Hits@1, Hits@3, Hits@10.
    """
    model.eval()
    all_mrr, all_h1, all_h3, all_h10 = [], [], [], []

    for batch in loader:
        batch = batch.to(device)

        node_emb, mask = model(batch)  # [B, N, H]

        # Score all candidate edges in this batch
        scores = score_edges(node_emb, batch.edge_label_index, batch.batch)
        labels = batch.edge_label.float()   # [E] binary

        # Per-graph MRR — group by graph
        # We need to know which edges belong to which graph
        # edge_label_index uses global node indices; use batch assignment
        src_nodes  = batch.edge_label_index[0]
        edge_batch = batch.batch[src_nodes]   # graph id for each candidate edge

        for g in edge_batch.unique():
            g_mask  = edge_batch == g
            g_scores = scores[g_mask]
            g_labels = labels[g_mask]

            pos_mask = g_labels == 1
            neg_mask = g_labels == 0

            if pos_mask.sum() == 0 or neg_mask.sum() == 0:
                continue

            pos_scores = g_scores[pos_mask]   # [P]
            neg_scores = g_scores[neg_mask]   # [Q]

            # For each positive, count how many negatives score higher
            # rank = 1 + number of negatives with higher score
            for ps in pos_scores:
                rank = 1 + (neg_scores > ps).sum().item()
                all_mrr.append(1.0 / rank)
                all_h1.append(1.0  if rank <= 1  else 0.0)
                all_h3.append(1.0  if rank <= 3  else 0.0)
                all_h10.append(1.0 if rank <= 10 else 0.0)

    return {
        'MRR':     float(np.mean(all_mrr)),
        'Hits@1':  float(np.mean(all_h1)),
        'Hits@3':  float(np.mean(all_h3)),
        'Hits@10': float(np.mean(all_h10)),
    }


print('Evaluation function defined.')

## Training Function

In [ ]:
def train_contact(
    model_kwargs,
    max_epochs   = 150,
    patience     = 20,
    batch_size   = 8,
    accum_steps  = 4,
    seeds        = [0, 1, 2],
):
    train_loader, val_loader, test_loader = load_pcqm_contact(batch_size=batch_size)

    results = []

    for seed in seeds:
        print('=' * 60)
        print(f'Dataset: PCQM-Contact | Seed: {seed}')
        print(f'Batch size: {batch_size} | Accum steps: {accum_steps} | Effective: {batch_size * accum_steps}')
        print('=' * 60)

        set_seed(seed)

        model     = GraphFNet_Contact(**model_kwargs).to(device)
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max_epochs, eta_min=1e-5
        )
        criterion = nn.BCEWithLogitsLoss()
        ckpt_path = f'best_graphfnet_contact_seed{seed}.pt'

        param_count = count_params(model)
        print(f'Parameters: {param_count:,}')
        print(f'Checkpoint: {ckpt_path}')

        best_val_mrr      = 0.0
        best_epoch        = 0
        epochs_no_improve = 0

        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()

        for epoch in range(1, max_epochs + 1):

            # ---- Training ----
            model.train()
            total_loss  = 0
            epoch_start = time.time()
            optimizer.zero_grad()

            pbar = tqdm(
                enumerate(train_loader),
                total=len(train_loader),
                desc=f'Seed {seed} | Epoch {epoch}',
                leave=False
            )

            for step, batch in pbar:
                batch = batch.to(device)

                node_emb, mask = model(batch)
                scores = score_edges(node_emb, batch.edge_label_index, batch.batch)
                labels = batch.edge_label.float()

                loss = criterion(scores, labels) / accum_steps
                loss.backward()
                total_loss += loss.item() * accum_steps

                if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                pbar.set_postfix({'Loss': f'{loss.item() * accum_steps:.4f}'})

            scheduler.step()
            avg_loss   = total_loss / len(train_loader)
            current_lr = scheduler.get_last_lr()[0]

            # ---- Validation (MRR only for speed) ----
            val_metrics = evaluate_contact(model, val_loader)
            val_mrr     = val_metrics['MRR']
            epoch_time  = time.time() - epoch_start

            print(
                f'Epoch {epoch:03d} | '
                f'Loss {avg_loss:.4f} | '
                f'Val MRR {val_mrr:.4f} | '
                f'LR {current_lr:.6f} | '
                f'Time {epoch_time:.2f}s'
            )

            if val_mrr > best_val_mrr:
                best_val_mrr      = val_mrr
                best_epoch        = epoch
                epochs_no_improve = 0
                torch.save(model.state_dict(), ckpt_path)
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= patience:
                print(f'Early stopping at epoch {epoch}.')
                break

        # ---- Single clean test evaluation ----
        print(f'\nLoading best checkpoint (epoch {best_epoch})...')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        test_metrics = evaluate_contact(model, test_loader)

        total_time = time.time() - start_time
        peak_mem   = torch.cuda.max_memory_allocated() / 1024**2

        print('-' * 60)
        print(f'Seed {seed} Results:')
        print(f'  Best Val MRR: {best_val_mrr:.4f}')
        print(f'  Test MRR:     {test_metrics["MRR"]:.4f}')
        print(f'  Test Hits@1:  {test_metrics["Hits@1"]:.4f}')
        print(f'  Test Hits@3:  {test_metrics["Hits@3"]:.4f}')
        print(f'  Test Hits@10: {test_metrics["Hits@10"]:.4f}')
        print(f'  Best Epoch:   {best_epoch}')
        print(f'  Total Time:   {total_time:.2f}s')
        print(f'  Peak Memory:  {peak_mem:.2f} MB')
        print('-' * 60)

        results.append(test_metrics)

        log_results_to_csv('results_pcqm_contact.csv', {
            'Dataset'        : 'PCQM-Contact',
            'Model'          : 'GraphFNet_Contact',
            'HiddenDim'      : model_kwargs.get('hidden_dim'),
            'NumLayers'      : model_kwargs.get('num_layers'),
            'Params'         : param_count,
            'Seed'           : seed,
            'BestValMRR'     : best_val_mrr,
            'TestMRR'        : test_metrics['MRR'],
            'TestHits1'      : test_metrics['Hits@1'],
            'TestHits3'      : test_metrics['Hits@3'],
            'TestHits10'     : test_metrics['Hits@10'],
            'BestEpoch'      : best_epoch,
            'TotalTime'      : total_time,
            'PeakMemoryMB'   : peak_mem,
            'MaxEpochs'      : max_epochs,
            'Patience'       : patience,
            'BatchSize'      : batch_size,
            'AccumSteps'     : accum_steps,
            'EffectiveBatch' : batch_size * accum_steps,
        })

    # ---- Summary ----
    mrr_scores = [r['MRR'] for r in results]
    h1_scores  = [r['Hits@1'] for r in results]
    h3_scores  = [r['Hits@3'] for r in results]
    h10_scores = [r['Hits@10'] for r in results]

    print('=' * 60)
    print(f'FINAL Results across {len(seeds)} seeds:')
    print(f'  MRR:     {np.mean(mrr_scores):.4f} ± {np.std(mrr_scores):.4f}')
    print(f'  Hits@1:  {np.mean(h1_scores):.4f} ± {np.std(h1_scores):.4f}')
    print(f'  Hits@3:  {np.mean(h3_scores):.4f} ± {np.std(h3_scores):.4f}')
    print(f'  Hits@10: {np.mean(h10_scores):.4f} ± {np.std(h10_scores):.4f}')
    print('=' * 60)

    return results


print('Training function defined.')

## Run Training

In [ ]:
results = train_contact(
    model_kwargs = {
        'hidden_dim' : 128,
        'num_layers' : 4,
        'num_heads'  : 4,
        'lap_k'      : 8,
        'dropout'    : 0.1,
        'edge_dim'   : 3,
    },
    max_epochs  = 150,
    patience    = 20,
    batch_size  = 8,
    accum_steps = 4,
    seeds       = [0, 1, 2],
)